# 08 Threshold Rule Optimization

基于验证集搜索正常类阈值，并对比原始预测、阈值调整、阈值调整加业务规则兜底三组结果。


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

for candidate in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    candidate_text = str(candidate)
    if candidate_text not in sys.path:
        sys.path.insert(0, candidate_text)

from src.models.model_evaluator import CreditModelEvaluator
from src.models.threshold_adjuster import RiskThresholdAdjuster

ARTIFACT_DIR = PROJECT_ROOT / "src" / "models" / "artifacts"
MODEL_DIR = PROJECT_ROOT / "src" / "models"
DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["正常类", "关注类", "次级类", "可疑类", "损失类"]


In [ ]:
class FixedPredictionModel:
    """Wrap fixed predictions so CreditModelEvaluator can score them."""

    def __init__(self, y_pred: np.ndarray, y_proba: np.ndarray) -> None:
        self._y_pred = np.asarray(y_pred, dtype=int)
        self._y_proba = np.asarray(y_proba, dtype=float)
        self.classes_ = np.arange(self._y_proba.shape[1])

    def predict(self, X):
        return self._y_pred

    def predict_proba(self, X):
        return self._y_proba


In [ ]:
preprocessor = joblib.load(ARTIFACT_DIR / "preprocessor.joblib")
selector = joblib.load(ARTIFACT_DIR / "selector.joblib")
risk_classifier = joblib.load(ARTIFACT_DIR / "risk_classifier.joblib")

val_df = pd.read_csv(resolve_split_path("val"), low_memory=False)
test_df = pd.read_csv(resolve_split_path("test"), low_memory=False)

y_val = val_df["preloan_risk_label"].astype(int)
y_test = test_df["preloan_risk_label"].astype(int)

X_val_processed = preprocessor.transform(val_df)
X_test_processed = preprocessor.transform(test_df)
X_val_selected = selector.transform(X_val_processed)
X_test_selected = selector.transform(X_test_processed)

val_proba = risk_classifier.predict_proba(X_val_selected)
test_proba = risk_classifier.predict_proba(X_test_selected)
raw_test_pred = risk_classifier.predict(X_test_selected)

display(Markdown(f"当前模型实际类别: `{list(getattr(risk_classifier, 'classes_', []))}`"))


In [ ]:
adjuster = RiskThresholdAdjuster(normal_class_threshold=0.9)
best_threshold, best_threshold_metrics = adjuster.find_optimal_threshold(
    y_val_proba=val_proba,
    y_val=y_val,
    min_normal_precision=0.8,
)

display(Markdown(f"最优正常类阈值: `{best_threshold:.2f}`"))
display(Markdown(json.dumps(best_threshold_metrics, ensure_ascii=False, indent=2)))


In [ ]:
def apply_business_rules(frame: pd.DataFrame, predictions: np.ndarray, adjuster: RiskThresholdAdjuster) -> np.ndarray:
    final_predictions = []
    for (_, row), prediction in zip(frame.iterrows(), predictions):
        final_predictions.append(adjuster.business_rule_override(row, int(prediction)))
    return np.asarray(final_predictions, dtype=int)

threshold_test_pred = adjuster.adjust_prediction(test_proba)
final_test_pred = apply_business_rules(test_df, threshold_test_pred, adjuster)

raw_model = FixedPredictionModel(raw_test_pred, test_proba)
threshold_model = FixedPredictionModel(threshold_test_pred, test_proba)
final_model = FixedPredictionModel(final_test_pred, test_proba)


In [ ]:
raw_metrics = CreditModelEvaluator(raw_model, X_test_selected, y_test, CLASS_NAMES).evaluate_imbalanced_multiclass()
threshold_metrics = CreditModelEvaluator(threshold_model, X_test_selected, y_test, CLASS_NAMES).evaluate_imbalanced_multiclass()
final_metrics = CreditModelEvaluator(final_model, X_test_selected, y_test, CLASS_NAMES).evaluate_imbalanced_multiclass()

comparison_df = pd.DataFrame(
    [
        {
            "组别": "原始模型预测",
            "Macro-F1": round(float(raw_metrics["macro_f1"]), 6),
            "Weighted-F1": round(float(raw_metrics["weighted_f1"]), 6),
            "平均KS": round(float(raw_metrics["ks_value"]), 6),
            "正常类精准率": round(float(raw_metrics["classification_report_named"].get("正常类", {}).get("precision", 0.0)), 6),
        },
        {
            "组别": "阈值调整后",
            "Macro-F1": round(float(threshold_metrics["macro_f1"]), 6),
            "Weighted-F1": round(float(threshold_metrics["weighted_f1"]), 6),
            "平均KS": round(float(threshold_metrics["ks_value"]), 6),
            "正常类精准率": round(float(threshold_metrics["classification_report_named"].get("正常类", {}).get("precision", 0.0)), 6),
        },
        {
            "组别": "阈值调整+业务规则",
            "Macro-F1": round(float(final_metrics["macro_f1"]), 6),
            "Weighted-F1": round(float(final_metrics["weighted_f1"]), 6),
            "平均KS": round(float(final_metrics["ks_value"]), 6),
            "正常类精准率": round(float(final_metrics["classification_report_named"].get("正常类", {}).get("precision", 0.0)), 6),
        },
    ]
)
display(comparison_df)


In [ ]:
threshold_grid = np.round(np.arange(0.50, 0.991, 0.01), 2)
normal_precision_series = []
high_risk_recall_series = []

for threshold in threshold_grid:
    adjuster.normal_class_threshold = float(threshold)
    val_pred = adjuster.adjust_prediction(val_proba)

    evaluator = CreditModelEvaluator(
        FixedPredictionModel(val_pred, val_proba),
        X_val_selected,
        y_val,
        CLASS_NAMES,
    )
    metrics = evaluator.evaluate_imbalanced_multiclass()
    report = metrics["classification_report_named"]

    normal_precision_series.append(float(report.get("正常类", {}).get("precision", 0.0)))
    risk_recalls = [
        float(report.get(class_name, {}).get("recall", 0.0))
        for class_name in metrics["class_names"]
        if class_name != "正常类"
    ]
    high_risk_recall_series.append(float(np.mean(risk_recalls)) if risk_recalls else 0.0)

adjuster.normal_class_threshold = float(best_threshold)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(threshold_grid, normal_precision_series, label="正常类精准率", linewidth=2)
ax.plot(threshold_grid, high_risk_recall_series, label="高风险类平均召回率", linewidth=2)
ax.axvline(best_threshold, color="gray", linestyle="--", label=f"最优阈值={best_threshold:.2f}")
ax.set_xlabel("正常类阈值")
ax.set_ylabel("Score")
ax.set_ylim(0.0, 1.0)
ax.set_title("阈值敏感性分析")
ax.grid(alpha=0.3, linestyle="--")
ax.legend()

output_path = FIGURE_DIR / "threshold_sensitivity.png"
fig.tight_layout()
fig.savefig(output_path, dpi=200, bbox_inches="tight")
plt.close(fig)

display(Markdown(f"阈值敏感性图已保存到 `results/figures/{output_path.name}`"))


In [ ]:
best_threshold_payload = {
    "best_threshold": float(best_threshold),
    "search_constraint": {
        "min_normal_precision": 0.8,
    },
    "best_threshold_metrics": best_threshold_metrics,
    "comparison_summary": comparison_df.to_dict(orient="records"),
    "class_names": raw_metrics["class_names"],
}

best_threshold_path = MODEL_DIR / "best_threshold_config.json"
best_threshold_path.write_text(
    json.dumps(best_threshold_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display(Markdown(f"最优阈值配置已保存到 `src/models/{best_threshold_path.name}`"))
